In [23]:
import pandas as pd
import requests
from io import BytesIO
import numpy as np
import kagglehub
import os
import re
from thefuzz import fuzz, process

In [24]:
realtor = pd.read_csv("https://econdata.s3-us-west-2.amazonaws.com/Reports/Core/RDC_Inventory_Core_Metrics_County_History.csv")
realtor_hotness = pd.read_csv("https://econdata.s3-us-west-2.amazonaws.com/Reports/Hotness/RDC_Inventory_Hotness_Metrics_County_History.csv")
realtor_merge = pd.merge(realtor, realtor_hotness, how="outer", on=["month_date_yyyymm", "county_fips"])

investor_data  = pd.read_csv("https://media.githubusercontent.com/media/walde363/covid-housing-forecast/main/data/raw/dowload_investor_purchases_by_metro.csv", encoding="utf-16", sep="\t")

BLS_unemployment = pd.read_csv("https://media.githubusercontent.com/media/walde363/covid-housing-forecast/main/data/raw/BLS_unemployment_data.csv")
BLS_weekly_earnings = pd.read_csv("https://media.githubusercontent.com/media/walde363/covid-housing-forecast/main/data/raw/BLS_weekly_earnings.csv")

processed_data = pd.read_csv("https://media.githubusercontent.com/media/walde363/covid-housing-forecast/main/data/processed/processed_data.csv")
Huricanes = processed_data[['date','highest_category']]
Huricanes['date'] = pd.to_datetime(Huricanes['date'])

url = "https://www.freddiemac.com/pmms/docs/historicalweeklydata.xlsx"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
response = requests.get(url, headers=headers)
response.raise_for_status()
mortgage_rates = pd.read_excel(BytesIO(response.content))

C:\Users\mduro\AppData\Local\Temp\ipykernel_38288\52829127.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Huricanes['date'] = pd.to_datetime(Huricanes['date'])


In [46]:
investor_df_clean = investor_data.copy(deep=True)

# Split year and quarter
investor_df_clean[["year", "q"]] = investor_df_clean["Quarter"].str.split(" ", expand=True)

investor_df_clean["year"] = investor_df_clean["year"].astype(int)
investor_df_clean["quarter"] = investor_df_clean["q"].str.replace("Q", "").astype(int)

# Convert quarter → starting month
investor_df_clean["month"] = (investor_df_clean["quarter"] - 1) * 3 + 1

# Create datetime (quarter start)
investor_df_clean["date"] = pd.to_datetime(
    dict(year=investor_df_clean["year"], month=investor_df_clean["month"], day=1)
)

investor_df_clean["Investor Purchases"] = (
    investor_df_clean["Investor Purchases"]
    .str.replace(",", "", regex=False)
    .astype("Int32")
)

investor_df_clean["Investor Market Share"] = (
    investor_df_clean["Investor Market Share"]
    .str.replace("%", "", regex=False)
    .astype("Int32")
)

investor_df_clean["Investor Purchases YoY"] = (
    investor_df_clean["Investor Purchases YoY"]
    .str.replace("%", "", regex=False)
    .astype("Int32")
)

investor_df_clean = investor_df_clean.groupby(["year", "month", "date", "Redfin Metro"]).agg(
    {"Investor Purchases": "sum"
    ,"Investor Market Share": "mean"
    ,"Investor Purchases YoY": "mean"}).reset_index()

investor_df_clean[['city', 'state']] = investor_df_clean['Redfin Metro'].str.split(',', expand=True)
investor_df_clean = investor_df_clean.groupby(["date","state"]).agg({"Investor Purchases": "sum", "Investor Market Share": "mean", "Investor Purchases YoY": "mean"}).reset_index()
investor_df_clean['state'] = investor_df_clean['state'].str.lower().str.strip()

In [47]:
mortgage_rates_df_clean = mortgage_rates.copy(deep=True)
mortgage_rates_df_clean = mortgage_rates_df_clean[mortgage_rates_df_clean['Unnamed: 0'].notnull()]

# take first row as header
mortgage_rates_df_clean.columns = mortgage_rates_df_clean.iloc[0]
mortgage_rates_df_clean = mortgage_rates_df_clean[1:].reset_index(drop=True)

week_parsed = pd.to_datetime(mortgage_rates_df_clean["Week"], errors="coerce")
mortgage_rates_df_clean = mortgage_rates_df_clean[week_parsed.notna()].copy()
mortgage_rates_df_clean["Week"] = week_parsed[week_parsed.notna()].dt.date

# renaming columns from base spreedsheet
mortgage_rates_df_clean.columns.values[0] = "Week"
mortgage_rates_df_clean.columns.values[1] = "U.S. 30 year FRM"
mortgage_rates_df_clean.columns.values[2] = "30 year fees & points"

mortgage_rates_df_clean.columns.values[3] = "U.S. 15 year FRM"
mortgage_rates_df_clean.columns.values[4] = "15 year fees & points"

mortgage_rates_df_clean.columns.values[5] = "U.S. 5/1 ARM"
mortgage_rates_df_clean.columns.values[6] = "5/1 year fees & points"

mortgage_rates_df_clean.columns.values[7] = "U.S. 5/1 ARM margin"
mortgage_rates_df_clean.columns.values[8] = "30 year FRM / 5/1 ARM spread"

#Truncate
mortgage_rates_df_clean = mortgage_rates_df_clean[mortgage_rates_df_clean['Week']>=pd.to_datetime("2016-01-01").date()]

mortgage_rates_df_clean['Week'] = pd.to_datetime(mortgage_rates_df_clean['Week'])
mortgage_rates_df_clean['Month'] = mortgage_rates_df_clean['Week'].dt.to_period('M').dt.to_timestamp()
mortgage_rates_df_clean.drop(columns=['Week'], inplace=True)
mortgage_rates_df_clean = mortgage_rates_df_clean.groupby('Month').mean().reset_index()

In [48]:
BLS_unemployment_data_df_clean = BLS_unemployment.copy(deep=True)

BLS_unemployment_data_df_clean = pd.melt(BLS_unemployment_data_df_clean,
                                 id_vars=["State","Area"],
                                 var_name="Month",
                                 value_name="Unemployment_Rate")

# statiscal area names were cleaned for join but there is not a 1 for 1 match. One example is below.
# Alachua is an example of a county not in earnings data
# BLS_weekly_earnings_df_clean[BLS_weekly_earnings_df_clean['State']=='Florida']['Area'].unique()

BLS_unemployment_data_df_clean['Area'] = BLS_unemployment_data_df_clean['Area'].str.replace(" Metropolitan Statistical Area", "", regex=False)

BLS_unemployment_data_df_clean = BLS_unemployment_data_df_clean[BLS_unemployment_data_df_clean['State'] != BLS_unemployment_data_df_clean['Area']]

In [49]:
BLS_weekly_earnings_df_clean = BLS_weekly_earnings.copy(deep=True)

BLS_weekly_earnings_df_clean['Area'] = np.where(
    BLS_weekly_earnings_df_clean['Area'] == 'Statewide',
    BLS_weekly_earnings_df_clean['State'],
    BLS_weekly_earnings_df_clean['Area']
)

BLS_weekly_earnings_df_clean = pd.melt(BLS_weekly_earnings_df_clean,
                                 id_vars=["State","Area"],
                                 var_name="Month",
                                 value_name="Earnings")

BLS_weekly_earnings_df_clean = BLS_weekly_earnings_df_clean[BLS_weekly_earnings_df_clean['State'] != BLS_weekly_earnings_df_clean['Area']]
BLS_weekly_earnings_df_clean = BLS_weekly_earnings_df_clean[~BLS_weekly_earnings_df_clean['Area'].str.contains('Metropolitan Division', na=False)]

In [50]:
BLS_df_clean_merge = pd.merge(BLS_weekly_earnings_df_clean,BLS_unemployment_data_df_clean,how="outer", on=["State","Area","Month"])
BLS_df_clean_merge[['city', 'state']] = BLS_df_clean_merge['Area'].str.split(', ', expand=True)
BLS_df_clean_merge['city'] = BLS_df_clean_merge['city'].str.lower()
BLS_df_clean_merge['state'] = BLS_df_clean_merge['state'].str.lower()
BLS_df_clean_merge['Month'] = pd.to_datetime(BLS_df_clean_merge['Month'].str.replace('M', '', regex=False).str.replace('-', '', regex=False), format='%Y%m')
BLS_df_clean_merge = BLS_df_clean_merge.groupby(["state","Month"]).agg({"Earnings": "mean", "Unemployment_Rate": "mean"}).reset_index()

In [51]:
realtor_merge = pd.merge(realtor, realtor_hotness, how="outer", on=["month_date_yyyymm", "county_fips"])
realtor_merge[['city', 'state']] = realtor_merge['county_name_x'].str.split(', ', expand=True)
realtor_merge['month_date_yyyymm'] = pd.to_datetime(realtor_merge['month_date_yyyymm'], format='%Y%m')

In [52]:
realtor_bls_merge = pd.merge(realtor_merge, BLS_df_clean_merge, how="left", left_on=["state", "month_date_yyyymm"], right_on=["state", "Month"])

In [53]:
#by quarter by state - resample to monthly, dividing values by 3 to preserve scale
numeric_cols = ['Investor Purchases', 'Investor Market Share', 'Investor Purchases YoY']
investor_df_clean[numeric_cols] = investor_df_clean[numeric_cols] / 3
investor_df_clean = investor_df_clean.set_index('date').groupby('state').resample('MS').ffill().drop(columns='state').reset_index()

In [54]:
processed_data_pre_model = realtor_bls_merge.copy(deep=True)
processed_data_pre_model = pd.merge(processed_data_pre_model, investor_df_clean, left_on=['month_date_yyyymm','state'], right_on=["date","state"], how="left")
processed_data_pre_model = pd.merge(processed_data_pre_model, mortgage_rates_df_clean, left_on="month_date_yyyymm", right_on="Month", how="left")
processed_data_pre_model = pd.merge(processed_data_pre_model, Huricanes, left_on="month_date_yyyymm", right_on="date", how="left")


In [55]:
processed_data_pre_model.to_csv("../data/processed/processed_data_pre_model.csv", index=False)

In [59]:
processed_data_pre_model.isna().sum().sort_values(ascending=False)

highest_category            341377
price_increased_count_mm    267360
price_increased_count_yy    264977
cbsa_code                   228316
cbsa_title                  228316
                             ...  
state                            0
city                             0
Month_y                          0
U.S. 30 year FRM                 0
U.S. 15 year FRM                 0
Length: 90, dtype: int64